# Lecture 29 - Dimensionality Reduction and Clustering

## Learning Objectives

- Apply PCA for dimensionality reduction and visualisation
- Interpret explained variance ratio and scree plots
- Use K-Means clustering and the elbow method
- Evaluate clusters with silhouette score
- Scale features before PCA and K-Means
- Use DBSCAN for density-based clustering

## Key Topics

- PCA and explained variance ratio
- Scree plots
- 2D visualisation after PCA
- K-Means and inertia
- Elbow method and silhouette score
- StandardScaler before PCA / K-Means
- DBSCAN for density-based clustering

## PCA: Principal Component Analysis

**PCA** is an unsupervised technique that reduces the dimensionality of data while preserving as much variance as possible. It finds new axes (principal components) that are linear combinations of the original features, ordered by how much variance they capture.

Why reduce dimensions?
- **Visualisation**: project high-dimensional data into 2D or 3D for plotting
- **Noise reduction**: discard low-variance components that may be noise
- **Feature compression**: reduce memory and computation for downstream models
- **Multicollinearity**: decorrelate features before linear models

The `explained_variance_ratio_` tells you the proportion of total variance captured by each component. A **scree plot** visualises this — look for the "elbow" where adding more components yields diminishing returns.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

digits = load_digits()
X_d, y_d = digits.data, digits.target
print(f"Original shape: {X_d.shape} (64 features)")

# Scale before PCA
X_scaled = StandardScaler().fit_transform(X_d)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"After PCA (2 components): {X_pca.shape}")

In [ ]:
# 2D visualisation after PCA
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_d, cmap="tab10",
                      alpha=0.7, s=40)
plt.colorbar(scatter, label="Digit")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
plt.title("PCA of Digits Dataset (64D -> 2D)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Scree plot: explained variance ratio
pca_full = PCA().fit(X_scaled)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
         np.cumsum(pca_full.explained_variance_ratio_), "bo-")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("Cumulative Explained Variance")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(range(1, 11), pca_full.explained_variance_ratio_[:10])
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Scree Plot (first 10 components)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## K-Means Clustering

**K-Means** is one of the most popular clustering algorithms. It partitions data into k clusters, where each point belongs to the cluster with the nearest mean (centroid).

The algorithm works iteratively: (1) assign each point to the nearest centroid, (2) recompute centroids as the mean of assigned points, (3) repeat until convergence.

Key concepts:
- **Inertia**: sum of squared distances of samples to their closest centroid. Lower is better, but it decreases monotonically with k, making it a poor standalone criterion for choosing k.
- **Elbow method**: plot inertia vs k and look for the "elbow" point where adding more clusters gives diminishing returns.
- **Silhouette score**: measures how similar a point is to its own cluster vs neighbouring clusters. Ranges from -1 to 1 — higher is better.

Always scale features before K-Means! Otherwise, features with larger scales dominate distance calculations.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# K-Means on the PCA-reduced digits data
kmeans = KMeans(n_clusters=10, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X_scaled)

print(f"Silhouette score (10 clusters): "
      f"{silhouette_score(X_scaled, labels):.3f}")

# Visualise clustering in 2D PCA space
plt.figure(figsize=(10, 7))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10",
            alpha=0.7, s=40)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=200, edgecolors="black", label="Centroids")
plt.title("K-Means Clustering on Digits (PCA-reduced)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Elbow method and silhouette score for optimal k
inertias = []
silhouettes = []
k_range = range(2, 15)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertias, "bo-")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouettes, "ro-")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(silhouettes)]
print(f"Optimal k by silhouette score: {best_k}")

## DBSCAN: Density-Based Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed together, marking points in low-density regions as outliers.

Unlike K-Means:
- You don't need to specify the number of clusters
- It can find arbitrarily shaped clusters
- It handles noise/outliers naturally

The algorithm has two parameters:
- `eps`: maximum distance between two points to be considered neighbours
- `min_samples`: minimum neighbours to form a dense region

DBSCAN is excellent for anomaly detection and datasets with clusters of irregular shapes, but struggles when clusters have vastly different densities.

In [ ]:
from sklearn.cluster import DBSCAN

# Generate data with clusters and outliers
from sklearn.datasets import make_blobs

X_blob, y_blob = make_blobs(n_samples=300, centers=3,
                            cluster_std=1.0, random_state=42)
# Add some outliers
rng = np.random.RandomState(42)
outliers = rng.uniform(low=-10, high=10, size=(30, 2))
X_blob = np.vstack([X_blob, outliers])

dbscan = DBSCAN(eps=1.5, min_samples=5)
db_labels = dbscan.fit_predict(X_blob)

n_clusters = len(set(db_labels) - {-1})
n_noise = list(db_labels).count(-1)
print(f"DBSCAN found {n_clusters} clusters and {n_noise} noise points")

In [ ]:
# Visualise DBSCAN results
plt.figure(figsize=(8, 6))
unique_labels = set(db_labels)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    mask = db_labels == label
    label_name = f"Noise (label={label})" if label == -1 else f"Cluster {label}"
    plt.scatter(X_blob[mask, 0], X_blob[mask, 1],
                c=[color], label=label_name, alpha=0.7, s=30, edgecolors="black")

plt.title(f"DBSCAN Clustering ({n_clusters} clusters, {n_noise} outliers)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Data Science Connection

Dimensionality reduction and clustering are the workhorses of unsupervised learning. PCA lets you visualise high-dimensional data and compress features before modelling — it is used in everything from genetics (thousands of genes down to 2D) to finance (factors from hundreds of stock returns). K-Means segments customers, DBSCAN detects credit card fraud. Together, these methods reveal the hidden structure in your data that you cannot see with raw numbers alone.